# Notebook 03: Cell Edge Detection

## Goal
Detect the edges of the first cell using local Hough transform with short line segments.

## Concepts Covered
1. **Region of Interest (ROI)** - Extract a small region for focused processing
2. **Canny edge detection** - Find edges in the ROI
3. **Hough Line Transform** - Detect line segments (HoughLinesP)
4. **Line classification** - Separate horizontal and vertical lines
5. **Line intersection** - Compute where lines meet

## Algorithm
```
1. Extract ROI around first cell with margin
2. Apply Canny edge detection on ROI
3. Run HoughLinesP with parameters for short lines
4. Classify lines: horizontal vs vertical
5. Select best top edge (horizontal) and left edge (vertical)
6. Compute intersection → refined corner
7. Use cell size to compute all 4 corners
```

## Setup

In [ ]:
import sys
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

import cv2
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import List, Dict, Optional, Tuple

# Our utilities
from utils.visualization import (
    show_image, show_images_grid, draw_quadrilateral, 
    draw_cell_highlight, draw_corners
)
from utils.geometry import (
    order_corners, line_angle, line_intersection, 
    classify_line_orientation, point_distance
)

# Sample image loading
from tests.border_detection.sampler import sample_images, load_image

print(f"Project root: {PROJECT_ROOT}")

## Load and Process Image (From Previous Notebooks)

We need the warped grid and first cell location from Notebooks 01 and 02.

In [ ]:
# Functions from previous notebooks
def find_outer_border(image):
    """Detect outer border using contour detection."""
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    binary = cv2.adaptiveThreshold(
        blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV, 11, 2
    )
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for contour in sorted(contours, key=cv2.contourArea, reverse=True):
        peri = cv2.arcLength(contour, True)
        approx = cv2.approxPolyDP(contour, 0.02 * peri, True)
        if len(approx) == 4:
            return order_corners(approx.reshape(4, 2))
    return None

def warp_to_top_down(image, corners, output_size=450):
    """Apply perspective transform."""
    src_pts = corners.astype(np.float32)
    dst_pts = np.array([
        [0, 0], [output_size-1, 0], 
        [output_size-1, output_size-1], [0, output_size-1]
    ], dtype=np.float32)
    matrix = cv2.getPerspectiveTransform(src_pts, dst_pts)
    return cv2.warpPerspective(image, matrix, (output_size, output_size))

@dataclass
class BlobInfo:
    label: int
    x: int
    y: int
    width: int
    height: int
    area: int
    centroid_x: float
    centroid_y: float
    
    @property
    def bbox(self): return (self.x, self.y, self.width, self.height)
    @property
    def aspect_ratio(self): return self.width / self.height if self.height > 0 else 0

def find_first_cell(warped_image):
    """Find the top-left cell in a warped grid."""
    gray = cv2.cvtColor(warped_image, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    binary = cv2.adaptiveThreshold(
        blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV, 11, 2
    )
    inverted = cv2.bitwise_not(binary)
    
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(inverted)
    
    image_area = warped_image.shape[0] * warped_image.shape[1]
    min_area = 0.003 * image_area
    max_area = 0.02 * image_area
    
    blobs = []
    for i in range(1, num_labels):
        area = stats[i, cv2.CC_STAT_AREA]
        w = stats[i, cv2.CC_STAT_WIDTH]
        h = stats[i, cv2.CC_STAT_HEIGHT]
        aspect = w / h if h > 0 else 0
        
        if min_area <= area <= max_area and 0.5 <= aspect <= 2.0:
            blobs.append(BlobInfo(
                label=i,
                x=stats[i, cv2.CC_STAT_LEFT],
                y=stats[i, cv2.CC_STAT_TOP],
                width=w, height=h, area=area,
                centroid_x=centroids[i, 0],
                centroid_y=centroids[i, 1]
            ))
    
    if not blobs:
        return None
    
    return min(blobs, key=lambda b: b.centroid_x + b.centroid_y)

In [ ]:
# Load and process sample image
sample_paths = sample_images(1, seed=42)
image_path = sample_paths[0]
original = load_image(image_path)

# Detect border and warp
corners = find_outer_border(original)
warped = warp_to_top_down(original, corners, output_size=450) if corners is not None else None

# Find first cell
first_cell = find_first_cell(warped) if warped is not None else None

if first_cell:
    print(f"First cell found at: ({first_cell.x}, {first_cell.y})")
    print(f"Size: {first_cell.width} x {first_cell.height}")
    
    # Visualize
    x, y, w, h = first_cell.bbox
    cell_corners = np.array([[x, y], [x+w, y], [x+w, y+h], [x, y+h]])
    vis = draw_cell_highlight(warped, cell_corners, fill_alpha=0.3)
    show_image(vis, "First Cell Located")
else:
    print("Failed to find first cell!")

---
## Step 1: Extract Region of Interest (ROI)

### Why ROI?
- **Focused processing**: Only analyze the relevant area
- **Better parameters**: Hough parameters can be tuned for cell-sized features
- **Reduced noise**: Less chance of detecting irrelevant lines

### Margin
We expand the bounding box slightly to ensure we capture the cell edges:
- Too small margin: Might miss edges
- Too large margin: Might include neighboring cell edges

In [ ]:
def extract_roi(
    image: np.ndarray,
    bbox: Tuple[int, int, int, int],
    margin_ratio: float = 0.3,
) -> Tuple[np.ndarray, Tuple[int, int]]:
    """
    Extract a region of interest with margin.
    
    Args:
        image: Input image
        bbox: Bounding box as (x, y, width, height)
        margin_ratio: Margin as fraction of bbox size
        
    Returns:
        Tuple of (roi_image, (roi_x, roi_y)) for offset tracking
    """
    x, y, w, h = bbox
    
    # Calculate margin
    margin_x = int(w * margin_ratio)
    margin_y = int(h * margin_ratio)
    
    # Expand bounding box
    x1 = max(0, x - margin_x)
    y1 = max(0, y - margin_y)
    x2 = min(image.shape[1], x + w + margin_x)
    y2 = min(image.shape[0], y + h + margin_y)
    
    # Extract ROI
    roi = image[y1:y2, x1:x2].copy()
    
    return roi, (x1, y1)

# Extract ROI around first cell
if first_cell and warped is not None:
    roi, (roi_x, roi_y) = extract_roi(warped, first_cell.bbox, margin_ratio=0.3)
    
    print(f"Original bbox: {first_cell.bbox}")
    print(f"ROI offset: ({roi_x}, {roi_y})")
    print(f"ROI shape: {roi.shape}")
    
    show_images_grid(
        [warped, roi],
        ["Full Warped Grid", "ROI Around First Cell"],
        cols=2
    )

---
## Step 2: Canny Edge Detection

### The Canny Algorithm
1. **Noise reduction**: Gaussian blur
2. **Gradient calculation**: Sobel operators for Gx, Gy
3. **Non-maximum suppression**: Thin edges to 1 pixel wide
4. **Hysteresis thresholding**:
   - Strong edges: gradient > high_threshold
   - Weak edges: low_threshold < gradient < high_threshold
   - Keep weak edges only if connected to strong edges

### Parameters
- `low_threshold`: Below this, not an edge
- `high_threshold`: Above this, definitely an edge
- Typical ratio: high = 2-3 × low

In [ ]:
def apply_canny(
    image: np.ndarray,
    low_threshold: int = 50,
    high_threshold: int = 150,
) -> np.ndarray:
    """
    Apply Canny edge detection.
    
    Args:
        image: Input image (grayscale or BGR)
        low_threshold: Lower threshold for hysteresis
        high_threshold: Upper threshold for hysteresis
        
    Returns:
        Binary edge image
    """
    # Convert to grayscale if needed
    if len(image.shape) == 3:
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    else:
        gray = image
    
    # Apply Gaussian blur
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    
    # Apply Canny
    edges = cv2.Canny(blurred, low_threshold, high_threshold)
    
    return edges

# Apply Canny to ROI
if first_cell and warped is not None:
    edges = apply_canny(roi, low_threshold=50, high_threshold=150)
    
    # Compare different thresholds
    edges_30_90 = apply_canny(roi, 30, 90)
    edges_50_150 = apply_canny(roi, 50, 150)
    edges_100_200 = apply_canny(roi, 100, 200)
    
    show_images_grid(
        [roi, edges_30_90, edges_50_150, edges_100_200],
        ["Original ROI", "Canny 30/90", "Canny 50/150", "Canny 100/200"],
        cols=4
    )

---
## Step 3: Hough Line Transform (HoughLinesP)

### The Hough Transform
Transforms edge points to a parameter space where lines become peaks:
- Each edge point (x, y) votes for all lines passing through it
- Lines are represented in polar form: ρ = x·cos(θ) + y·sin(θ)
- Peaks in the accumulator = detected lines

### HoughLinesP (Probabilistic)
More efficient than standard Hough:
- Randomly samples edge pixels
- Returns **line segments** [x1, y1, x2, y2], not infinite lines
- Parameters:
  - `threshold`: Minimum votes to be considered a line
  - `minLineLength`: Minimum segment length
  - `maxLineGap`: Maximum gap between points in a line

In [ ]:
def detect_lines_hough(
    edges: np.ndarray,
    threshold: int = 30,
    min_line_length: int = 20,
    max_line_gap: int = 5,
) -> np.ndarray:
    """
    Detect line segments using probabilistic Hough transform.
    
    Args:
        edges: Binary edge image
        threshold: Minimum votes for a line
        min_line_length: Minimum line segment length
        max_line_gap: Maximum gap to connect segments
        
    Returns:
        Array of lines, each as [x1, y1, x2, y2]
    """
    lines = cv2.HoughLinesP(
        edges,
        rho=1,
        theta=np.pi / 180,
        threshold=threshold,
        minLineLength=min_line_length,
        maxLineGap=max_line_gap,
    )
    
    if lines is None:
        return np.array([])
    
    return lines.reshape(-1, 4)

# Detect lines in the ROI
if first_cell and warped is not None:
    # Use cell size to set parameters
    cell_size = (first_cell.width + first_cell.height) / 2
    min_length = int(cell_size * 0.3)  # Lines should be at least 30% of cell size
    
    lines = detect_lines_hough(
        edges,
        threshold=20,
        min_line_length=min_length,
        max_line_gap=5
    )
    
    print(f"Cell size: {cell_size:.1f} px")
    print(f"Min line length: {min_length} px")
    print(f"Detected {len(lines)} lines")

In [ ]:
def visualize_lines(image: np.ndarray, lines: np.ndarray, colors=None) -> np.ndarray:
    """Draw lines on an image."""
    vis = image.copy()
    if len(vis.shape) == 2:
        vis = cv2.cvtColor(vis, cv2.COLOR_GRAY2BGR)
    
    for i, line in enumerate(lines):
        x1, y1, x2, y2 = line
        color = colors[i] if colors else (0, 255, 0)
        cv2.line(vis, (x1, y1), (x2, y2), color, 2)
    
    return vis

# Visualize all detected lines
if len(lines) > 0:
    lines_vis = visualize_lines(roi, lines)
    
    show_images_grid(
        [roi, edges, lines_vis],
        ["ROI", "Edges", f"Lines ({len(lines)})"],
        cols=3
    )

---
## Step 4: Classify Lines (Horizontal vs Vertical)

### Line Angle
For a line segment from (x1, y1) to (x2, y2):
```
angle = atan2(y2 - y1, x2 - x1)
```

### Classification
- **Horizontal**: angle ≈ 0° or ≈ 180° (within threshold)
- **Vertical**: angle ≈ 90° (within threshold)
- **Diagonal**: Everything else (we ignore these)

In [ ]:
def classify_lines(
    lines: np.ndarray,
    angle_threshold: float = 20.0,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Classify lines as horizontal or vertical.
    
    Args:
        lines: Array of lines [x1, y1, x2, y2]
        angle_threshold: Degrees from horizontal/vertical to classify
        
    Returns:
        Tuple of (horizontal_lines, vertical_lines)
    """
    horizontal = []
    vertical = []
    
    for line in lines:
        angle = line_angle(line)
        
        # Horizontal: angle near 0° or 180°
        if angle < angle_threshold or angle > (180 - angle_threshold):
            horizontal.append(line)
        # Vertical: angle near 90°
        elif (90 - angle_threshold) < angle < (90 + angle_threshold):
            vertical.append(line)
    
    return np.array(horizontal), np.array(vertical)

# Classify lines
if len(lines) > 0:
    h_lines, v_lines = classify_lines(lines, angle_threshold=20.0)
    
    print(f"Horizontal lines: {len(h_lines)}")
    print(f"Vertical lines: {len(v_lines)}")
    
    # Print angles for inspection
    print("\nHorizontal line angles:")
    for line in h_lines[:5]:
        print(f"  {line_angle(line):.1f}°")
    
    print("\nVertical line angles:")
    for line in v_lines[:5]:
        print(f"  {line_angle(line):.1f}°")

In [ ]:
# Visualize classified lines
if len(h_lines) > 0 or len(v_lines) > 0:
    vis = roi.copy()
    if len(vis.shape) == 2:
        vis = cv2.cvtColor(vis, cv2.COLOR_GRAY2BGR)
    
    # Draw horizontal lines in red
    for line in h_lines:
        x1, y1, x2, y2 = line
        cv2.line(vis, (x1, y1), (x2, y2), (0, 0, 255), 2)
    
    # Draw vertical lines in blue
    for line in v_lines:
        x1, y1, x2, y2 = line
        cv2.line(vis, (x1, y1), (x2, y2), (255, 0, 0), 2)
    
    show_image(vis, f"Classified Lines: {len(h_lines)} H (red), {len(v_lines)} V (blue)")

---
## Step 5: Select Best Cell Edges

### Strategy
We're looking for the **top edge** (horizontal) and **left edge** (vertical) of the cell:
- **Top edge**: Horizontal line with smallest Y coordinate
- **Left edge**: Vertical line with smallest X coordinate

We use the line's midpoint for comparison since it's more stable than endpoints.

In [ ]:
def get_line_position(line: np.ndarray, is_horizontal: bool) -> float:
    """
    Get the relevant position of a line (Y for horizontal, X for vertical).
    Uses midpoint for stability.
    """
    x1, y1, x2, y2 = line
    
    if is_horizontal:
        return (y1 + y2) / 2  # Average Y position
    else:
        return (x1 + x2) / 2  # Average X position


def select_cell_edges(
    h_lines: np.ndarray,
    v_lines: np.ndarray,
) -> Tuple[Optional[np.ndarray], Optional[np.ndarray]]:
    """
    Select the top (horizontal) and left (vertical) edges.
    
    Args:
        h_lines: Horizontal line segments
        v_lines: Vertical line segments
        
    Returns:
        Tuple of (top_edge, left_edge) or (None, None)
    """
    top_edge = None
    left_edge = None
    
    # Find topmost horizontal line (smallest Y)
    if len(h_lines) > 0:
        top_edge = min(h_lines, key=lambda l: get_line_position(l, is_horizontal=True))
    
    # Find leftmost vertical line (smallest X)
    if len(v_lines) > 0:
        left_edge = min(v_lines, key=lambda l: get_line_position(l, is_horizontal=False))
    
    return top_edge, left_edge

# Select cell edges
if len(h_lines) > 0 and len(v_lines) > 0:
    top_edge, left_edge = select_cell_edges(h_lines, v_lines)
    
    if top_edge is not None:
        print(f"Top edge: {top_edge}")
        print(f"  Position: Y = {get_line_position(top_edge, True):.1f}")
    
    if left_edge is not None:
        print(f"Left edge: {left_edge}")
        print(f"  Position: X = {get_line_position(left_edge, False):.1f}")

In [ ]:
# Visualize selected edges
if top_edge is not None and left_edge is not None:
    vis = roi.copy()
    if len(vis.shape) == 2:
        vis = cv2.cvtColor(vis, cv2.COLOR_GRAY2BGR)
    
    # Draw top edge in green
    x1, y1, x2, y2 = top_edge
    cv2.line(vis, (x1, y1), (x2, y2), (0, 255, 0), 3)
    cv2.putText(vis, "TOP", (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 0), 1)
    
    # Draw left edge in cyan
    x1, y1, x2, y2 = left_edge
    cv2.line(vis, (x1, y1), (x2, y2), (255, 255, 0), 3)
    cv2.putText(vis, "LEFT", (x1 - 30, y1), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 0), 1)
    
    show_image(vis, "Selected Cell Edges")

---
## Step 6: Compute Line Intersection

### The Math
Two lines in parametric form:
```
Line 1: P1 + t * (P2 - P1)
Line 2: P3 + s * (P4 - P3)
```

Solving for the intersection gives us the top-left corner of the cell.

In [ ]:
# Compute intersection of top and left edges
if top_edge is not None and left_edge is not None:
    intersection = line_intersection(top_edge, left_edge)
    
    if intersection:
        ix, iy = intersection
        print(f"Intersection (in ROI coords): ({ix:.1f}, {iy:.1f})")
        
        # Convert to full image coordinates
        ix_full = ix + roi_x
        iy_full = iy + roi_y
        print(f"Intersection (in warped image coords): ({ix_full:.1f}, {iy_full:.1f})")
        
        # Compare with blob-detected cell position
        print(f"\nBlob-detected cell corner: ({first_cell.x}, {first_cell.y})")
        print(f"Difference: ({ix_full - first_cell.x:.1f}, {iy_full - first_cell.y:.1f})")
    else:
        print("Lines are parallel - no intersection!")

In [ ]:
# Visualize intersection
if intersection:
    vis = roi.copy()
    if len(vis.shape) == 2:
        vis = cv2.cvtColor(vis, cv2.COLOR_GRAY2BGR)
    
    # Draw edges
    x1, y1, x2, y2 = top_edge
    cv2.line(vis, (x1, y1), (x2, y2), (0, 255, 0), 2)
    
    x1, y1, x2, y2 = left_edge
    cv2.line(vis, (x1, y1), (x2, y2), (255, 255, 0), 2)
    
    # Draw intersection point
    ix_int, iy_int = int(ix), int(iy)
    cv2.circle(vis, (ix_int, iy_int), 6, (0, 0, 255), -1)
    cv2.putText(vis, "CORNER", (ix_int + 10, iy_int - 10), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 255), 1)
    
    show_image(vis, "Top-Left Corner via Line Intersection")

---
## Step 7: Compute Cell Boundaries

Using the intersection point and estimated cell size, we can compute all 4 corners.

In [ ]:
def compute_cell_corners(
    top_left: Tuple[float, float],
    cell_width: float,
    cell_height: float,
) -> np.ndarray:
    """
    Compute all 4 corners of a cell from top-left corner.
    
    Args:
        top_left: (x, y) of top-left corner
        cell_width: Width of the cell
        cell_height: Height of the cell
        
    Returns:
        4x2 array of corners [TL, TR, BR, BL]
    """
    x, y = top_left
    
    return np.array([
        [x, y],                          # TL
        [x + cell_width, y],             # TR
        [x + cell_width, y + cell_height],  # BR
        [x, y + cell_height],            # BL
    ], dtype=np.float32)

# Compute cell corners
if intersection:
    # Use blob-detected size as estimate
    cell_width = first_cell.width
    cell_height = first_cell.height
    
    # Convert intersection to full image coords
    top_left = (ix + roi_x, iy + roi_y)
    
    cell_corners = compute_cell_corners(top_left, cell_width, cell_height)
    
    print("Computed cell corners:")
    labels = ["TL", "TR", "BR", "BL"]
    for label, corner in zip(labels, cell_corners):
        print(f"  {label}: ({corner[0]:.1f}, {corner[1]:.1f})")

In [ ]:
# Final visualization - highlight the detected cell on the warped grid
if intersection and warped is not None:
    final_vis = draw_cell_highlight(
        warped, 
        cell_corners.astype(np.int32),
        fill_color=(0, 255, 0),
        fill_alpha=0.3,
        border_color=(0, 255, 0),
        border_thickness=2
    )
    
    # Draw the refined corner point
    cv2.circle(final_vis, (int(top_left[0]), int(top_left[1])), 5, (0, 0, 255), -1)
    
    # Add label
    cv2.putText(
        final_vis, 
        "FIRST CELL (Edge-Detected)", 
        (10, 30),
        cv2.FONT_HERSHEY_SIMPLEX, 
        0.6, 
        (0, 255, 0), 
        2
    )
    
    show_image(final_vis, "Final Result: First Cell Detected via Local Hough")

---
## Complete Pipeline Function

In [ ]:
@dataclass
class EdgeDetectionResult:
    """Result from cell edge detection."""
    success: bool
    cell_corners: Optional[np.ndarray] = None
    top_edge: Optional[np.ndarray] = None
    left_edge: Optional[np.ndarray] = None
    intersection: Optional[Tuple[float, float]] = None
    roi: Optional[np.ndarray] = None
    roi_offset: Optional[Tuple[int, int]] = None
    debug_images: Optional[Dict[str, np.ndarray]] = None


def detect_cell_edges(
    warped_image: np.ndarray,
    cell_bbox: Tuple[int, int, int, int],
    margin_ratio: float = 0.3,
    canny_low: int = 50,
    canny_high: int = 150,
    hough_threshold: int = 20,
    angle_threshold: float = 20.0,
    return_debug: bool = False,
) -> EdgeDetectionResult:
    """
    Detect cell edges using local Hough transform.
    
    Args:
        warped_image: Top-down view of Sudoku grid
        cell_bbox: Bounding box of the cell (x, y, w, h)
        margin_ratio: ROI margin as fraction of cell size
        canny_low: Canny low threshold
        canny_high: Canny high threshold
        hough_threshold: Hough line votes threshold
        angle_threshold: Degrees from H/V to classify
        return_debug: Whether to return debug images
        
    Returns:
        EdgeDetectionResult with detected cell boundaries
    """
    debug = {}
    x, y, w, h = cell_bbox
    
    # Step 1: Extract ROI
    roi, (roi_x, roi_y) = extract_roi(warped_image, cell_bbox, margin_ratio)
    if return_debug:
        debug["1_roi"] = roi.copy()
    
    # Step 2: Canny edges
    edges = apply_canny(roi, canny_low, canny_high)
    if return_debug:
        debug["2_edges"] = edges
    
    # Step 3: Hough lines
    min_length = int(min(w, h) * 0.3)
    lines = detect_lines_hough(edges, hough_threshold, min_length, max_line_gap=5)
    if return_debug:
        debug["3_lines"] = visualize_lines(roi, lines)
    
    if len(lines) == 0:
        return EdgeDetectionResult(
            success=False, roi=roi, roi_offset=(roi_x, roi_y),
            debug_images=debug if return_debug else None
        )
    
    # Step 4: Classify lines
    h_lines, v_lines = classify_lines(lines, angle_threshold)
    
    if len(h_lines) == 0 or len(v_lines) == 0:
        return EdgeDetectionResult(
            success=False, roi=roi, roi_offset=(roi_x, roi_y),
            debug_images=debug if return_debug else None
        )
    
    # Step 5: Select best edges
    top_edge, left_edge = select_cell_edges(h_lines, v_lines)
    
    # Step 6: Compute intersection
    intersection = line_intersection(top_edge, left_edge)
    
    if intersection is None:
        return EdgeDetectionResult(
            success=False, top_edge=top_edge, left_edge=left_edge,
            roi=roi, roi_offset=(roi_x, roi_y),
            debug_images=debug if return_debug else None
        )
    
    # Step 7: Compute cell corners
    top_left = (intersection[0] + roi_x, intersection[1] + roi_y)
    cell_corners = compute_cell_corners(top_left, w, h)
    
    if return_debug:
        result_vis = draw_cell_highlight(
            warped_image, cell_corners.astype(np.int32)
        )
        debug["4_result"] = result_vis
    
    return EdgeDetectionResult(
        success=True,
        cell_corners=cell_corners,
        top_edge=top_edge,
        left_edge=left_edge,
        intersection=top_left,
        roi=roi,
        roi_offset=(roi_x, roi_y),
        debug_images=debug if return_debug else None,
    )

In [ ]:
# Test the complete pipeline
if first_cell and warped is not None:
    result = detect_cell_edges(warped, first_cell.bbox, return_debug=True)
    
    print(f"Success: {result.success}")
    if result.success:
        print(f"Cell corners:\n{result.cell_corners}")
    
    if result.debug_images:
        show_images_grid(
            list(result.debug_images.values()),
            list(result.debug_images.keys()),
            cols=4
        )

---
## Test on Multiple Images

In [ ]:
# Test on 6 sample images
test_paths = sample_images(6, seed=456)
test_images = [load_image(p) for p in test_paths]

result_images = []
success_count = 0

for i, (path, img) in enumerate(zip(test_paths, test_images)):
    # Full pipeline: border -> warp -> first cell -> edge detection
    border = find_outer_border(img)
    
    if border is None:
        vis = img.copy()
        cv2.putText(vis, "No border", (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
        result_images.append(cv2.resize(vis, (200, 200)))
        continue
    
    warped = warp_to_top_down(img, border, output_size=450)
    cell = find_first_cell(warped)
    
    if cell is None:
        vis = warped.copy()
        cv2.putText(vis, "No cell", (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
        result_images.append(cv2.resize(vis, (200, 200)))
        continue
    
    result = detect_cell_edges(warped, cell.bbox)
    
    if result.success:
        success_count += 1
        vis = draw_cell_highlight(warped, result.cell_corners.astype(np.int32))
        cv2.putText(vis, "SUCCESS", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
    else:
        vis = warped.copy()
        x, y, w, h = cell.bbox
        cv2.rectangle(vis, (x, y), (x+w, y+h), (0, 0, 255), 2)
        cv2.putText(vis, "Edge fail", (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
    
    result_images.append(cv2.resize(vis, (200, 200)))

print(f"Success rate: {success_count}/{len(test_images)}")

titles = [f"{p.name[:12]}..." for p in test_paths]
show_images_grid(result_images, titles, cols=3)

---
## Summary

### What We Learned
1. **ROI extraction** focuses processing on the relevant area
2. **Canny edge detection** finds edges using gradient and hysteresis thresholding
3. **HoughLinesP** detects line segments by voting in parameter space
4. **Line classification** separates horizontal and vertical lines by angle
5. **Line intersection** finds where two lines meet using parametric form

### The Hough Transform Formula
```
ρ = x·cos(θ) + y·sin(θ)

where:
  ρ = perpendicular distance from origin to line
  θ = angle of the perpendicular
```

### Key Parameters
| Parameter | Default | Purpose |
|-----------|---------|--------|
| `margin_ratio` | 0.3 | ROI expansion around cell |
| `canny_low/high` | 50/150 | Edge detection sensitivity |
| `hough_threshold` | 20 | Minimum line votes |
| `min_line_length` | 30% of cell | Filter short segments |
| `angle_threshold` | 20° | H/V classification tolerance |

### Complete Algorithm
```
1. Detect outer border (Notebook 01)
2. Perspective transform to top-down view
3. Find first cell via blob detection (Notebook 02)
4. Extract ROI with margin
5. Canny edge detection
6. HoughLinesP for line segments
7. Classify lines as H/V
8. Select top and left edges
9. Compute intersection → cell corner
10. Highlight the detected cell!
```

### Future Extensions
- Propagate to detect all 81 cells using the cell template
- Use detected grid for OCR digit extraction
- Handle cases where edge detection fails (fallback to blob bounds)